<a href="https://colab.research.google.com/github/Dineshseervi/AI_IIITM/blob/main/week_15/cross_encoding_BM_23_02_patterns.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# W15 - Legal Contract Q&A: Patterns

**Level:** Intermediate -> Advanced
**Stack:** LangChain v1 + FAISS + rank-bm25 + Cohere Rerank + Together AI
**Domain:** Legal / Contract Q&A
**Time:** ~110 min

## Learning objectives

By the end of this notebook, you can:
- Implement hybrid search combining BM25 (lexical) and FAISS (semantic) with reciprocal rank fusion
- Add Cohere Rerank as a post-retrieval re-scoring layer (and explain bi-encoder vs cross-encoder)
- Apply three query rewriting patterns: HyDE, multi-query expansion, step-back prompting
- Implement Anthropic's contextual retrieval (chunk-summary prefix before embedding)
- Reason about which technique helps which kind of failure mode

This is the closure of the largest gap in modern LLM curricula. Every technique here ships in real RAG systems at scale.

## Roadmap

**Section 1 - Hybrid search (BM25 + semantic with RRF)**
- 1.1 The retrieval problem with naive RAG (recap from 01)
- 1.2 What is BM25? Lexical scoring intuition
- 1.3 BM25 implementation with `rank-bm25`
- 1.4 What is semantic search? Embeddings recap
- 1.5 Reciprocal Rank Fusion - what it is, why we use it
- 1.6 Hybrid search demo (BM25 + semantic with RRF)

**Section 2 - Reranking**
- 2.1 What is reranking? Cross-encoder vs bi-encoder
- 2.2 Cohere Rerank demo (graceful skip if no key)
- 2.3 Why reranking dramatically improves quality

**Section 3 - Query rewriting**
- 3.1 Query rewriting overview
- 3.2 HyDE (hypothetical document embeddings)
- 3.3 Multi-query expansion
- 3.4 Step-back prompting

**Section 4 - Contextual retrieval**
- 4.1 Anthropic contextual retrieval (the 49%/67%/10x technique)
- 4.2 Why prepending context helps

**Section 5 - Full pipeline composition**


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install langchain_together langchain_text_splitters langchain_community langchain_huggingface faiss-cpu sentence-transformers rank-bm25

In [ ]:
# Imports
from pathlib import Path
from typing import Optional
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from rank_bm25 import BM25Okapi
import os, re, math

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Walk up from this notebook to find _global/.env
def _load_env():
    cwd = Path.cwd()
    for parent in [cwd] + list(cwd.parents):
        env_path = parent / ".env"
        if env_path.exists():
            load_dotenv(env_path)
            return env_path
    # fallback to current directory
    load_dotenv()
    return None

_env_loaded = _load_env()
print(f".env loaded from: {_env_loaded}")

PROVIDER = os.getenv("LLM_PROVIDER", "together").lower()

if PROVIDER == "together":
    from langchain_together import ChatTogether
    llm = ChatTogether(
        model=os.getenv("LLM_MODEL_DEFAULT", "Qwen/Qwen2.5-7B-Instruct-Turbo"),
        temperature=0.3,
        max_tokens=600,
    )
elif PROVIDER == "openai":
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(
        model=os.getenv("LLM_MODEL_OPENAI", "gpt-4o-mini"),
        temperature=0.3,
    )
else:
    raise ValueError(f"Unknown LLM_PROVIDER: {PROVIDER}")

print(f"Provider: {PROVIDER}")


.env loaded from: /content/.env
Provider: together


## Setup check

If the cell above raised an error:
- Make sure `_global/.env` exists with `TOGETHER_API_KEY` set
- Run `python _global/scripts/verify_setup.py` from the repo root
- See `_global/README.md` for setup instructions


## Setup - rebuild the corpus

Same load -> chunk -> embed -> index pipeline as `01_principles`. We bundle it into one cell so you can run this notebook standalone.

In [ ]:
# Reload contracts and chunks (same as 01_principles)
def _find_data_dir():
    from pathlib import Path

    # Candidate roots in priority order
    candidates = [
        Path.cwd(),
        Path("/content/drive/MyDrive/GenAI-AgenticAI"),  # Colab + Drive
        Path("/content/drive/MyDrive"),
    ]

    for base in candidates:
        if not base.exists():
            continue
        for parent in [base] + list(base.parents):
            # Strategy 1: Week_15_Production_RAG/data
            d = parent / "Week_15_Production_RAG" / "data"
            if d.exists():
                return d

            # Strategy 2: data/ alongside 01_principles.ipynb
            d = parent / "data"
            if d.exists() and (parent / "01_principles.ipynb").exists():
                return d

    raise FileNotFoundError(
        "Couldn't find data directory under any known path.\n"
        "Expected: /content/drive/MyDrive/GenAI-AgenticAI/Week_15_Production_RAG/data"
    )
DATA_DIR = _find_data_dir()

contracts = [
    Document(page_content=p.read_text(encoding="utf-8"), metadata={"source": p.name})
    for p in sorted(DATA_DIR.glob("*.txt"))
]

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
chunks = splitter.split_documents(contracts)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
vectorstore = FAISS.from_documents(chunks, embeddings)
parser = StrOutputParser()
print(f"Loaded {len(chunks)} chunks across {len(contracts)} contracts.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded 12 chunks across 4 contracts.


## Section 1 - Hybrid search

### 1.1 The retrieval problem with naive RAG

Recall the four failure modes from the end of `01_principles`:

1. **Lexical-only queries** - users type defined terms ("Section 5") or rare entities ("BluePeak") that *exact-string* match would find but semantic similarity might miss.
2. **Paraphrased questions** - questions phrased very differently from how the contract phrases the answer.
3. **Multi-hop questions** - need passages from multiple documents.
4. **Long-tail entities** - numbers, codes, names that don't dominate the embedding.

**Hybrid search** addresses problems 1 and 2 in one move: combine an exact-string scorer (BM25) with a meaning-based scorer (embeddings) and take the best of both.

### 1.2 What is BM25? Lexical scoring intuition

**BM25** (Best Matching 25) is a 1990s information-retrieval algorithm that's still the default lexical scorer in Elasticsearch, Lucene, and most search engines. Three intuitions:

- **Term frequency** - if a query word appears more times in a document, the document is more relevant. (But with diminishing returns - 10 mentions isn't 10x better than 1.)
- **Inverse document frequency** - if a query word is *rare* across the whole corpus (e.g., "indemnification"), matches on that word count more than matches on common words ("the", "and").
- **Document length normalization** - long documents shouldn't get unfair credit just because they have room for more matches.

BM25 has zero learning - it's a closed-form formula on word counts. That's the strength: **it sees exact strings**. If a user types `"Section 5"`, BM25 will find documents containing that exact phrase, even if no embedding model would have considered them similar to a question about Section 5.

The cost: **it sees only exact strings**. "termination" and "ending the contract" are unrelated to BM25 unless one shares words with the other.

### 1.3 BM25 implementation with `rank-bm25`

`rank-bm25` is a tiny pure-Python implementation. Three steps:

1. **Tokenize** every chunk - lowercase + split on word characters. (Production search engines use real linguistic tokenizers; for our 4-contract demo, regex is fine.)
2. Build the `BM25Okapi` index over tokenized chunks. This computes the IDF table and stores per-document term frequencies.
3. To search: tokenize the query, call `bm25.get_scores(query_tokens)`, sort the indices by score.

In [ ]:
def tokenize(text: str) -> list[str]:
    """Lowercase + extract word characters. Sufficient for English contracts."""
    return re.findall(r"\w+", text.lower())

# Build BM25 index over the same chunks
chunk_texts = [c.page_content for c in chunks]
chunk_tokens = [tokenize(t) for t in chunk_texts]
bm25 = BM25Okapi(chunk_tokens)


def bm25_search(query: str, k: int = 10) -> list[Document]:
    scores = bm25.get_scores(tokenize(query))
    # argsort descending, take top k
    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [chunks[i] for i in top_idx]

print("BM25 index built and bm25_search() defined.")

BM25 index built and bm25_search() defined.


**What just happened:** we tokenized every chunk and built a BM25 index. `bm25_search("indemnification clause", k=3)` would now return the 3 chunks where those exact words (and other rare query words) appear most prominently, weighted by IDF and length-normalized.

### 1.4 What is semantic search? Embeddings recap

**Semantic search** is what we built in `01_principles`. The query is embedded into a 384-dim vector; documents are embedded the same way; cosine similarity (= dot product of normalized vectors) ranks them.

The strength: **it sees meaning**. Asking "how can either party end this agreement?" finds the "termination" clause even though no query word appears in the answer.

The cost: **it can miss exact-string matches**. A query for `"Section 5"` may rank a paragraph *about* Section 5's topic above the paragraph *labeled* Section 5, because the embedding model has no special respect for literal labels.

In [ ]:
def semantic_search(query: str, k: int = 10) -> list[Document]:
    return vectorstore.similarity_search(query, k=k)

print("semantic_search() defined.")

semantic_search() defined.


### 1.5 Reciprocal Rank Fusion - what it is, why we use it

Now we have two ranked lists: one from BM25, one from semantic. How do we combine them?

**Naive idea:** average the scores. Doesn't work - BM25 scores and cosine scores are on completely different scales, and the relationship is non-linear.

**Better idea: Reciprocal Rank Fusion (RRF).** Ignore the raw scores entirely. Use only the *rank* (position in each list).

The RRF formula for a document `d`:

```
RRF(d) = sum over each ranker:  1 / (k + rank_in_that_list(d))
```

Where `k` is a smoothing constant (60 is the conventional default). Higher RRF wins.

**Why this works:**
- A document at rank 1 contributes `1/61 ≈ 0.0164`.
- A document at rank 10 contributes `1/70 ≈ 0.0143`.
- A document at rank 100 contributes `1/160 ≈ 0.0063`.
- A document not in a list contributes 0 from that list.

So showing up high in *either* list helps; showing up high in *both* lists helps a lot. The constant `k=60` flattens the curve so that the difference between rank 1 and rank 5 isn't enormous - this prevents one list from dominating just because its top hit is unanimous.

**RRF is parameter-free and battle-tested.** TREC and Pinecone both default to RRF for hybrid fusion. It generalizes to any number of input lists - throw in a third source later (e.g., metadata-filtered results) and the merge logic doesn't change.

### 1.6 Hybrid search demo

Below: `reciprocal_rank_fusion` takes a list of ranked lists and returns the merged top-k. Then `hybrid_search` is just "BM25 top-N, semantic top-N, RRF the two".

Then we test on two contrasting queries:
- **Exact-phrase query**: BM25 should win.
- **Paraphrased query**: semantic should win.

Hybrid should track the winner in each case.

In [ ]:
def reciprocal_rank_fusion(
    ranked_lists: list[list[Document]],
    k: int = 60,
    top_k: int = 5,
) -> list[Document]:
    """Merge multiple ranked lists into one using RRF."""
    scores: dict[str, float] = {}
    docs_by_id: dict[str, Document] = {}
    for ranked in ranked_lists:
        for rank, doc in enumerate(ranked, start=1):
            # Use page_content as ID (assume unique chunks)
            doc_id = doc.page_content
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
            docs_by_id[doc_id] = doc
    sorted_ids = sorted(scores.keys(), key=lambda d: scores[d], reverse=True)
    return [docs_by_id[i] for i in sorted_ids[:top_k]]


def hybrid_search(query: str, k_each: int = 10, top_k: int = 5) -> list[Document]:
    bm = bm25_search(query, k=k_each)
    sem = semantic_search(query, k=k_each)
    return reciprocal_rank_fusion([bm, sem], top_k=top_k)


# Test: BM25 wins on exact phrases, semantic wins on paraphrases
query_exact = "indemnification clause"
query_para = "if one party harms the other, who pays"

print(f"--- {query_exact} ---")
print("BM25 top-3:")
for d in bm25_search(query_exact, k=3):
    print(f"  [{d.metadata['source']}] {d.page_content}")
print("Semantic top-3:")
for d in semantic_search(query_exact, k=3):
    print(f"  [{d.metadata['source']}] {d.page_content}")
print("Hybrid top-3:")
for d in hybrid_search(query_exact, top_k=3):
    print(f"  [{d.metadata['source']}] {d.page_content}")

--- indemnification clause ---
BM25 top-3:
  [contract_2_license.txt] 4. FEES
License fee is INR 50,00,000 payable in 12 equal monthly installments. Maintenance fee
is 18% of license fee annually.

5. WARRANTY
Licensor warrants the Software will perform substantially as documented for 90 days from
delivery. NO OTHER WARRANTIES, EXPRESS OR IMPLIED.

6. INDEMNIFICATION
Licensor shall indemnify Licensee against third-party intellectual property infringement
claims, capped at 100% of fees paid in the prior 12 months.

7. LIMITATION OF LIABILITY
Licensor's total liability shall not exceed the license fees paid in the prior 12 months.
No party shall be liable for lost profits, data loss, or consequential damages.

8. CONFIDENTIALITY
Both parties shall protect Confidential Information for 7 years post-termination.
  [contract_1_msa.txt] 4. CONFIDENTIALITY
Each party shall maintain the other's Confidential Information in strict confidence for
a period of 5 years following disclosure.

5. INDEM

In [ ]:
print(f"\n--- {query_para} ---")
print("BM25 top-3:")
for d in bm25_search(query_para, k=3):
    print(f"  [{d.metadata['source']}] {d.page_content[:120]}...")
print("Semantic top-3:")
for d in semantic_search(query_para, k=3):
    print(f"  [{d.metadata['source']}] {d.page_content[:120]}...")
print("Hybrid top-3:")
for d in hybrid_search(query_para, top_k=3):
    print(f"  [{d.metadata['source']}] {d.page_content[:120]}...")


--- if one party harms the other, who pays ---
BM25 top-3:
  [contract_3_nda.txt] NON-DISCLOSURE AGREEMENT
Between: Aldridge Capital Partners LLP ("Disclosing Party") and Helios Robotics Pvt Ltd
("Recei...
  [contract_4_employment.txt] 4. CONFIDENTIALITY AND IP ASSIGNMENT
Employee assigns all work-product IP to the Company. Confidentiality obligations su...
  [contract_1_msa.txt] 4. CONFIDENTIALITY
Each party shall maintain the other's Confidential Information in strict confidence for
a period of 5...
Semantic top-3:
  [contract_1_msa.txt] 8. GOVERNING LAW
This Agreement is governed by the laws of India. Disputes shall be resolved by arbitration
in Bengaluru...
  [contract_3_nda.txt] 4. EXCEPTIONS
Confidential Information does not include information that (a) is publicly known;
(b) was lawfully receive...
  [contract_4_employment.txt] 8. DISPUTE RESOLUTION
Disputes shall be resolved by arbitration in Hyderabad per the Arbitration and
Conciliation Act, 1...
Hybrid top-3:
  [contract_3_

**What just happened:** notice how BM25 dominates the exact-phrase query (it sees the literal word "indemnification") and semantic dominates the paraphrase query (it understands "harms"/"pays" relate to indemnification). Hybrid takes the best of both.

**Takeaway:** hybrid is rarely worse than either pure approach and often visibly better. **RRF is the production default** - parameter-free, scales to any number of lists, and consistently lifts recall without extra tuning.

## Section 2 - Reranking

### 2.1 What is reranking? Cross-encoder vs bi-encoder

The retrievers we have so far (FAISS and BM25) are **bi-encoders**:

- The query is encoded *independently* of any document.
- Each document is encoded *independently* of any query.
- Similarity is a cheap scalar operation (dot product) over precomputed vectors.

That's what makes them fast: you can pre-embed millions of documents once and answer queries in milliseconds. But there's a price - because the encoder never sees query and document *together*, it has no way to model query-specific nuance ("is this passage actually answering THIS question?").

A **cross-encoder** is the opposite trade. It takes (query, document) as a single concatenated input and outputs a single relevance score:

```
cross_encoder("indemnification cap?", "Section 5: ... twelve months of fees ...") -> 0.93
cross_encoder("indemnification cap?", "Section 1: scope of services ...")        -> 0.04
```

The model can attend to both at once - it can notice that "twelve months of fees" matches "cap" semantically. Result: much higher precision than bi-encoders. The cost: it cannot pre-compute - every (query, document) pair is a forward pass at query time. Too slow for the *whole* corpus, but perfect for re-scoring the top-20 from a fast bi-encoder.

**The two-stage architecture (production standard):**

```
Stage 1: hybrid bi-encoder retrieval -> top 20 candidates    (fast, broad)
Stage 2: cross-encoder rerank -> top 5                       (slow, precise)
```

### 2.2 Cohere Rerank demo (graceful skip if no key)

**Cohere Rerank** is the production-popular hosted cross-encoder. Free tier exists. We gracefully skip if `COHERE_API_KEY` is not set so the notebook still runs end-to-end without it.

The setup:
1. Check for the API key.
2. Try to import `langchain_cohere`. (It's an optional dep.)
3. Construct `CohereRerank(model="rerank-english-v3.0", top_n=5)`.
4. Use it as `reranker.compress_documents(documents=candidates, query=query)` - input is a list of `Document`s, output is a re-sorted list.

In [ ]:
COHERE_AVAILABLE = bool(os.getenv("COHERE_API_KEY"))
print(f"Cohere available: {COHERE_AVAILABLE}")

reranker = None
if COHERE_AVAILABLE:
    try:
        from langchain_cohere import CohereRerank
        reranker = CohereRerank(model="rerank-english-v3.0", top_n=5)
        print("Reranker initialized.")
    except ImportError:
        print("langchain_cohere not installed. pip install langchain-cohere")
        COHERE_AVAILABLE = False

In [ ]:
def hybrid_then_rerank(query: str, retrieve_k: int = 20, top_k: int = 5) -> list[Document]:
    """Hybrid retrieve top-20, then Cohere rerank to top-5."""
    candidates = hybrid_search(query, k_each=retrieve_k, top_k=retrieve_k)
    if not COHERE_AVAILABLE:
        # Graceful fallback: just return the top of the hybrid candidates
        return candidates[:top_k]
    reranked = reranker.compress_documents(documents=candidates, query=query)
    return list(reranked)[:top_k]


# Compare: hybrid-only top-5 vs hybrid+rerank top-5 on a tricky query
tricky_query = "What happens if the licensee tries to take apart the software?"
print(f"Query: {tricky_query}\n")

print("Hybrid top-5:")
for i, d in enumerate(hybrid_search(tricky_query, top_k=5), 1):
    print(f"  {i}. [{d.metadata['source']}] {d.page_content[:120]}...")

print("\nHybrid + Rerank top-5:")
for i, d in enumerate(hybrid_then_rerank(tricky_query), 1):
    print(f"  {i}. [{d.metadata['source']}] {d.page_content[:120]}...")

### 2.3 Why reranking dramatically improves quality

The query "what happens if the licensee tries to take apart the software?" is a paraphrase of "reverse engineer / decompile / disassemble" (which appears verbatim in the License agreement, Section 2). The chunk we want exists; we just need it ranked first.

**With hybrid alone**, several chunks might mention "licensee" or "software" without actually addressing the *consequences* of taking it apart. They could be ranked above the right chunk just because they match more keywords.

**With reranking**, the cross-encoder reads the query and each candidate together. It sees that "Section 2 of the License agreement" is the chunk that *answers* the question - not just one that mentions the same nouns. The right chunk gets pulled to position 1.

**Anthropic's published numbers:** combining contextual retrieval (which we cover in Section 4) with hybrid search and reranking reduced retrieval failures by **roughly 10x** vs naive RAG. Reranking alone typically delivers 20-40% precision gains.

**If you don't have a Cohere key:** the cell falls back to the hybrid output. The system still works, just with one less precision lift.

**Open-source alternatives:** `BAAI/bge-reranker-v2-m3` (free, runs locally on GPU/CPU) or `cross-encoder/ms-marco-MiniLM-L-6-v2` (smaller, CPU-friendly). Cohere is convenient because it's hosted and English-tuned for legal/business text.

## Section 3 - Query rewriting

### 3.1 Query rewriting overview

Sometimes the user's question doesn't match how the answer is phrased in the source. *Rewrite the query* before retrieving. Three patterns we'll cover:

| Pattern | What it does | Best when |
|---|---|---|
| **HyDE** | Generate a hypothetical answer; embed *that* | Answers in corpus look very different from questions |
| **Multi-query** | Generate N paraphrases; merge their results | User phrasing is idiosyncratic |
| **Step-back** | Generalize the question to retrieve broader context | Question is so specific no chunk has the exact terms |

All three involve one extra LLM call before retrieval. That's a real cost (latency + tokens) - choose them when retrieval quality is your bottleneck.

### 3.2 HyDE - hypothetical document embeddings

**The intuition:** in the corpus, *answers* are written like answers (declarative sentences with specifics: "The license term is 36 months."). *Questions* are written like questions ("How long is the license term?"). The embedding spaces of these two sentence styles are not identical - even when they're about the same topic.

**HyDE's fix:** ask the LLM to *guess* what the answer looks like, in the form of a contract sentence, then embed *that hypothetical answer* and retrieve from the vector store. The embedding now matches "answer-shaped" text - which is exactly what we want to find in the corpus.

**Counterintuitive:** even when the LLM's hypothetical is factually wrong, it still tends to retrieve better than the raw question, because it has the *shape* of the answer. The wrong details don't hurt the embedding - they're a tiny fraction of the vector's geometry.

In [ ]:
HYDE_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Given a legal question, produce ONE concise sentence that would plausibly appear "
     "in a contract as the answer. Be specific. Don't add caveats. Even if you don't know, "
     "make a reasonable guess."),
    ("human", "Question: {question}"),
])
hyde_chain = HYDE_PROMPT | llm | parser

def hyde_retrieve(question: str, k: int = 5) -> list[Document]:
    # 1. LLM generates a hypothetical answer-shaped sentence
    hypo = hyde_chain.invoke({"question": question})
    # 2. Embed and retrieve using the HYPOTHETICAL, not the question
    return vectorstore.similarity_search(hypo, k=k)


q = "How long must the receiving party keep secrets after our NDA ends?"
print(f"Question: {q}")
print(f"\nHyDE hypothetical: {hyde_chain.invoke({'question': q})}")
print("\nHyDE-retrieved top-3:")
for d in hyde_retrieve(q, k=3):
    print(f"  [{d.metadata['source']}] {d.page_content[:150]}...")

**What just happened:** the LLM produced a hypothetical answer like "Confidentiality obligations survive termination for X additional years." We embedded *that sentence* and retrieved. The retriever finds the actual NDA Section 5 ("Confidentiality obligations survive termination for 5 additional years") because it now matches answer-shape, not question-shape.

**Takeaway:** HyDE is the cheapest single-trick query rewrite. One extra LLM call, no other changes.

### 3.3 Multi-query expansion

**The intuition:** users phrase questions differently. "When can either side end this agreement?" "How do we exit the contract?" "What's the termination procedure?" are all the same question, but they may retrieve different chunks because each phrasing weights different concepts.

**Multi-query's fix:** ask the LLM to produce N paraphrases. Retrieve the top-k for each paraphrase (and the original). Merge with RRF. The chunks that *consistently* rank well across paraphrases bubble to the top.

**Why RRF here too:** same reason as hybrid search - each paraphrase produces its own ranking, and we want to fuse them without comparing absolute scores.

In [ ]:
MULTIQ_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Generate exactly 3 paraphrased versions of the user's question. Output ONLY the 3 "
     "paraphrases, one per line, no numbering, no extra text."),
    ("human", "Question: {question}"),
])
multiq_chain = MULTIQ_PROMPT | llm | parser

def multi_query_retrieve(question: str, k_per_query: int = 4, top_k: int = 5) -> list[Document]:
    # 1. Generate paraphrases
    paraphrases_raw = multiq_chain.invoke({"question": question})
    paraphrases = [p.strip() for p in paraphrases_raw.split("\n") if p.strip()][:3]
    # 2. Retrieve for the original + each paraphrase
    queries = [question] + paraphrases
    ranked = [vectorstore.similarity_search(q, k=k_per_query) for q in queries]
    # 3. RRF merge
    return reciprocal_rank_fusion(ranked, top_k=top_k)


q = "When can either side end the master agreement?"
print(f"Question: {q}\n")
paraphrases = multiq_chain.invoke({"question": q})
print(f"Paraphrases:\n{paraphrases}\n")
print("Multi-query top-3:")
for d in multi_query_retrieve(q, top_k=3):
    print(f"  [{d.metadata['source']}] {d.page_content[:150]}...")

**What just happened:** the LLM produced 3 paraphrases (e.g., "How can the parties terminate the master services agreement?"). We retrieved with each, RRF-merged. The "Termination" section of the MSA shows up at the top because it's relevant to *all* paraphrases - whereas a chunk that incidentally matched one paraphrase via a stray keyword gets diluted.

**Cost note:** multi-query makes N+1 retrieval calls (cheap) plus 1 LLM call for paraphrasing (more expensive than retrieval). Use it when retrieval is the bottleneck, not generation.

### 3.4 Step-back prompting

**The intuition:** sometimes a question is so specific that *no chunk* has the exact terms. "What is the indemnification cap in the MSA between Acme and BluePeak?" mentions specific party names that may not appear together in any single chunk.

**Step-back's fix:** before retrieving, ask the LLM for a *more general* version of the question. Retrieve for *both* the general and the specific - the general one pulls broader context, the specific one anchors on key terms. RRF the two.

**Example:**
- Specific: "What is the indemnification cap in the MSA between Acme and BluePeak?"
- Step-back: "What are typical indemnification clauses in master services agreements?"

The step-back query retrieves the actual indemnification *clause* (which doesn't mention Acme/BluePeak) because it's about the topic. The specific query may still pull useful chunks; RRF combines.

In [ ]:
STEPBACK_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Restate the user's specific question as a more general question. The general question "
     "should be answerable by retrieving broader context. Output only the general question."),
    ("human", "Specific question: {question}"),
])
stepback_chain = STEPBACK_PROMPT | llm | parser

def stepback_retrieve(question: str, k: int = 5) -> list[Document]:
    general = stepback_chain.invoke({"question": question}).strip()
    # Retrieve for both - the general gets broader context, the specific gets exact
    general_hits = vectorstore.similarity_search(general, k=k)
    specific_hits = vectorstore.similarity_search(question, k=k)
    return reciprocal_rank_fusion([general_hits, specific_hits], top_k=k)


q = "What's the indemnification cap in the MSA?"
print(f"Question: {q}")
print(f"Step-back: {stepback_chain.invoke({'question': q})}")
print("\nStep-back retrieved top-3:")
for d in stepback_retrieve(q, k=3):
    print(f"  [{d.metadata['source']}] {d.page_content[:150]}...")

**What just happened:** the LLM produced a generalized question ("What are typical indemnification provisions in MSAs?"). We retrieved for both and merged. The actual indemnification clause from `contract_1_msa.txt` ranks at the top.

**Takeaway: the three rewriting techniques optimize for different failure modes:**

| Technique | Best when | LLM calls | Retrieval calls |
|---|---|---|---|
| HyDE | Question phrased very differently from how source answers | 1 | 1 |
| Multi-query | User phrasing is idiosyncratic / multiple valid paraphrases | 1 | N+1 |
| Step-back | Question is so specific that no chunk has the exact terms | 1 | 2 |

In a real system, you might A/B test these or even *route* between them based on question characteristics (a lightweight classifier deciding which to use).

## Section 4 - Anthropic contextual retrieval

### 4.1 The 49% / 67% / 10x technique

[Anthropic's contextual retrieval blog post (2024)](https://www.anthropic.com/news/contextual-retrieval) showed a simple modification that dramatically lifts retrieval quality. The numbers are striking:

| Setup | Failure rate reduction |
|---|---|
| Contextual embeddings alone | -49% |
| Contextual embeddings + contextual BM25 | -67% |
| Above + reranking (top-20 -> top-5) | -10x (a tenth as many failures) |

**The technique:** for each chunk, ask the LLM *"given the parent document, write 1-2 sentences explaining what context this chunk is in."* Prepend that to the chunk text **before embedding**.

The chunks become **self-describing**. The chunk about "Section 6" no longer says just "Section 6"; it says: *"This is Section 6 of the Master Services Agreement between Acme Cloud Services and BluePeak Industries; it discusses limitation of liability."* Now the embedding encodes both the local content *and* the document-level context.

### 4.2 Why prepending context helps

**The retrieval failure mode it fixes:**

Without context, two chunks that say "the cap is twelve months of fees" - one from the MSA, one from the License - are nearly identical in embedding space. A query about "the MSA's cap" can retrieve either. After prepending context, the MSA chunk becomes "[CONTEXT: This is the indemnification section of the MSA between Acme and BluePeak.]\nthe cap is twelve months of fees" - now its embedding clearly belongs to the MSA neighborhood, not the License neighborhood.

**The cost:**

One LLM call per chunk *at indexing time*. For a 10K-chunk corpus, that's a one-time ~$1-5 indexing cost (with a cheap model) - paid once, durable improvement on every subsequent query.

**Why it composes with reranking:**

Contextual chunks make the bi-encoder retrieve better candidates. Reranking then re-scores those candidates with full query-document attention. Each step's gains stack.

We'll demonstrate on a few chunks rather than re-indexing everything (LLM cost), but the pattern is identical for full-scale ingestion.

In [ ]:
CONTEXT_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are summarizing a chunk of a contract. In ONE sentence, describe the contract this "
     "chunk belongs to and what this chunk is about. Output the sentence only - no preamble."),
    ("human", "FULL CONTRACT (truncated):\n{document}\n\nCHUNK:\n{chunk}"),
])
context_chain = CONTEXT_PROMPT | llm | parser


def add_context_prefix(chunk: Document, parent: Document) -> Document:
    """Prepend a 1-sentence context summary to the chunk."""
    parent_truncated = parent.page_content[:2000]
    context_sentence = context_chain.invoke({
        "document": parent_truncated,
        "chunk": chunk.page_content,
    }).strip()
    new_content = f"[CONTEXT: {context_sentence}]\n\n{chunk.page_content}"
    return Document(page_content=new_content, metadata={**chunk.metadata, "has_context": True})


# Demo: take the first chunk of contract 1, see what the context prefix adds
sample_chunk = chunks[0]
sample_parent = next(c for c in contracts if c.metadata["source"] == sample_chunk.metadata["source"])
contextual_chunk = add_context_prefix(sample_chunk, sample_parent)
print("Original chunk preview:")
print(sample_chunk.page_content[:300])
print("\nContextual chunk preview:")
print(contextual_chunk.page_content[:400])

In [ ]:
# Build a small contextual index over the first 8 chunks (real systems re-embed everything)
contextual_chunks = []
parent_by_source = {c.metadata["source"]: c for c in contracts}
for chunk in chunks[:8]:  # First 8 chunks for demo
    parent = parent_by_source[chunk.metadata["source"]]
    contextual_chunks.append(add_context_prefix(chunk, parent))

contextual_store = FAISS.from_documents(contextual_chunks, embeddings)

# Compare: vanilla retrieval vs contextual retrieval on a generic query
q = "what's the liability cap"
print(f"Query: {q}\n")
print("Vanilla (no context prefix) - top 2:")
for d in vectorstore.similarity_search(q, k=2):
    print(f"  [{d.metadata['source']}] {d.page_content[:140]}...")

print("\nContextual top-2:")
for d in contextual_store.similarity_search(q, k=2):
    src = d.metadata['source']
    print(f"  [{src}] {d.page_content[:200]}...")

**What just happened:** each chunk's embedding now incorporates a 1-sentence summary saying which contract it's from and what topic it covers. For a generic query like "what's the liability cap", the contextual store can disambiguate whether the user wants the MSA cap or the License cap - because both are now self-labeled in their embeddings.

**Practical recipe:**
1. Index time: for each chunk, generate a context sentence with a *cheap* model (Haiku, gpt-4o-mini, Qwen-7B - quality matters less here than throughput).
2. Prepend `[CONTEXT: ...]` to the chunk text.
3. Embed and store both as the vector index *and* as the BM25 corpus.
4. At query time, no change - retrieve as usual. The contextual prefixes do the work.

## Section 5 - Full pipeline composition

### 5.1 The production stack

The production pattern stacks all of these:

```
Query
  -> rewrite (multi-query OR HyDE OR step-back; pick one or all)
  -> hybrid retrieve top-20 (BM25 + semantic over contextual chunks)
  -> rerank to top-5
  -> generate with refusal pattern
```

In `03_industrial.ipynb` we wire this exact stack into a `LegalRAG` class with Pydantic models, citation enforcement, and RAGAS evaluation. For now, here's a one-shot composition you can use as a reference template.

The function below stacks: multi-query rewrite -> hybrid retrieve (per query) -> RRF merge -> rerank if Cohere -> generate.

In [ ]:
def production_rag(question: str, top_k: int = 5) -> tuple[str, list[Document]]:
    """Reference pipeline: multi-query rewrite -> hybrid retrieve -> rerank -> generate."""
    # Step 1: rewrite
    paraphrases_raw = multiq_chain.invoke({"question": question})
    paraphrases = [p.strip() for p in paraphrases_raw.split("\n") if p.strip()][:3]
    queries = [question] + paraphrases

    # Step 2: hybrid retrieve for each query, merge with RRF
    all_lists = []
    for q in queries:
        all_lists.append(bm25_search(q, k=10))
        all_lists.append(semantic_search(q, k=10))
    candidates = reciprocal_rank_fusion(all_lists, top_k=20)

    # Step 3: rerank if Cohere is available
    if COHERE_AVAILABLE:
        reranked = reranker.compress_documents(documents=candidates, query=question)
        final = list(reranked)[:top_k]
    else:
        final = candidates[:top_k]

    # Step 4: generate
    rag_prompt = ChatPromptTemplate.from_messages([
        ("system",
         "Answer the question using ONLY the provided context. If the context doesn't contain "
         "the answer, say \"I don't have enough information.\" Quote phrases verbatim where possible."),
        ("human", "CONTEXT:\n{context}\n\nQUESTION:\n{question}"),
    ])
    context = "\n\n".join(f"[{d.metadata['source']}]\n{d.page_content}" for d in final)
    answer = (rag_prompt | llm | parser).invoke({"context": context, "question": question})
    return answer, final


# Test
q = "How is confidential information defined and how long does the obligation last in the NDA?"
ans, sources = production_rag(q)
print(f"Q: {q}\n")
print(f"A: {ans}\n")
print(f"Sources used: {[s.metadata['source'] for s in sources]}")

## Recap

You now have the full toolkit:

| Technique | Section | What it fixes |
|---|---|---|
| **Hybrid search (BM25 + semantic + RRF)** | 1 | One pattern, covers exact-string + paraphrase failures |
| **Cohere Rerank (cross-encoder)** | 2 | Ranks the right chunk to position 1 |
| **HyDE** | 3.2 | Question/answer phrasing mismatch |
| **Multi-query** | 3.3 | Idiosyncratic user phrasing |
| **Step-back** | 3.4 | Over-specific questions |
| **Contextual retrieval** | 4 | Chunks become self-describing |

`03_industrial.ipynb` ties this into a class-based `LegalRAG` system, adds Pydantic citations enforced by `with_structured_output`, RAGAS evaluation on a 10-question eval set, a Graph RAG sketch, and a production decision table.
